In [ ]:
import os
import numpy as np
import yaml
import matplotlib.pyplot as plt
import healpy as hp
import heracles
import heracles.dices as dices
from heracles.io import read
from astropy.io import fits
from scipy.ndimage import gaussian_filter1d

font = {'size'   : 16}
import matplotlib
matplotlib.rc('font', **font)
matplotlib.rc('xtick', labelsize=12) 
matplotlib.rc('ytick', labelsize=12) 

# Comparison

In [ ]:
config_path = "scripts/sims_config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)
n = 200
rcond = 0.05
nside = 2048
lmin = config['lmin']
lmax_full = 2000 #config['lmax_full']
lmax_partial = 4000 #config['lmax_partial']
lmax_mask = 6000 #config['lmax_mask']  # Default to lmax if not specified
mode = "gatti" #config['mode']  # "lognormal" or "gaussian"
mask_type = config['mask_type']  # Default to 'dr1' if not specified
binned = False #True
apply_mask = True
opt_method_nu = "naive"
opt_method_inv = "naive"
nbins = 3

output_path = f"{mode}_dices/"
output_path = "./masked_"+output_path

nlbins = config.get('nlbins', 20)  # Default to 20 if not specified
l_partial = np.arange(lmax_partial+1)
l_full = np.arange(lmax_full+1)
l_mask = np.arange(lmax_mask+1)
ledges = np.logspace(np.log10(lmin), np.log10(lmax_full), nlbins + 1)
lgrid = (ledges[1:] + ledges[:-1]) / 2
ledges_mask = np.logspace(np.log10(lmin), np.log10(lmax_mask), nlbins + 1)
lgrid_mask = (ledges_mask[1:] + ledges_mask[:-1]) / 2

## Nzs

In [ ]:
path = f"/pscratch/sd/j/jaimerz/{mode}_sims"

# Load nzs
nzs = np.load(f"{path}/nzs.npz")
z = nzs['z']
nz_1 = nzs['nz_1']
nz_2 = nzs['nz_2']

In [ ]:
plt.plot(z, nz_1, label="Lenses")
plt.plot(z, nz_2, label="Sources")
plt.xlabel(r"$z$")
plt.ylabel(r"$n(z)$")
plt.xlim(0,3)
plt.legend()

plt.savefig("./plots/nzs.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Masks

In [ ]:
sim = hp.read_map(f"/pscratch/sd/j/jaimerz/{mode}_sims/{mode}_sim_3_nside_{nside}/POS_1.fits")
mask_patch = hp.read_map(f"/pscratch/sd/j/jaimerz/masks/patch_mask_nside_{nside}.fits")
mask_dr1 = hp.read_map(f"/pscratch/sd/j/jaimerz/masks/dr1_mask_nside_{nside}.fits")
mask_tr1 = hp.read_map(f"/pscratch/sd/j/jaimerz/masks/tr1_mask_nside_{nside}.fits")

In [ ]:
fsky_dr1=np.sum(mask_dr1)/len(mask_dr1)
fsky_tr1=np.sum(mask_tr1)/len(mask_tr1)
fsky_patch=np.sum(mask_patch)/len(mask_patch)

In [ ]:
print(41252.96*fsky_dr1)
print(41252.96*fsky_tr1)
print(41252.96*fsky_patch)

In [ ]:
import numpy as np
import healpy as hp
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ---------- Helpers ----------
def mask_com_vec(mask, nside):
    pix = np.where(mask > 0)[0]
    vx, vy, vz = hp.pix2vec(nside, pix)
    v = np.array([vx.mean(), vy.mean(), vz.mean()], dtype=float)
    return v / np.linalg.norm(v)

def rotation_matrix_a_to_b(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)

    v = np.cross(a, b)
    c = np.dot(a, b)
    s = np.linalg.norm(v)

    if s < eps:
        if c > 0:
            return np.eye(3)
        axis = np.array([1.0, 0.0, 0.0])
        if abs(np.dot(axis, a)) > 0.9:
            axis = np.array([0.0, 1.0, 0.0])
        v = np.cross(a, axis)
        v = v / np.linalg.norm(v)
        K = np.array([[0, -v[2], v[1]],
                      [v[2], 0, -v[0]],
                      [-v[1], v[0], 0]])
        return np.eye(3) + 2.0 * (K @ K)

    K = np.array([[0, -v[2], v[1]],
                  [v[2], 0, -v[0]],
                  [-v[1], v[0], 0]])
    return np.eye(3) + K + K @ K * ((1.0 - c) / (s**2))

def rotate_binary_mask(mask, nside, R):
    pix = np.where(mask > 0)[0]
    vx, vy, vz = hp.pix2vec(nside, pix)
    vec = np.vstack([vx, vy, vz])
    vec_r = R @ vec
    pix_r = hp.vec2pix(nside, vec_r[0], vec_r[1], vec_r[2])
    out = np.zeros_like(mask, dtype=float)
    out[pix_r] = 1.0
    return out

# ---------- 1) Rotate patch COM -> COM(dr1 U tr1) ----------
target_mask = ((mask_dr1 > 0) | (mask_tr1 > 0)).astype(float)
v_target = mask_com_vec(target_mask, nside)
v_patch = mask_com_vec(mask_patch, nside)

R = rotation_matrix_a_to_b(v_patch, v_target)
mask_patch_rot = rotate_binary_mask(mask_patch, nside, R)

# ---------- 2) Crop region from joint occupied area ----------
union_mask = ((mask_dr1 > 0) | (mask_tr1 > 0) | (mask_patch_rot > 0)).astype(float)
pix_u = np.where(union_mask > 0)[0]
ux, uy, uz = hp.pix2vec(nside, pix_u)
uvec = np.vstack([ux, uy, uz]).T

v_center = mask_com_vec(union_mask, nside)
lon0 = float(np.degrees(np.arctan2(v_center[1], v_center[0])))
lat0 = float(np.degrees(np.arcsin(np.clip(v_center[2], -1.0, 1.0))))
rot = (lon0, lat0, 0.0)

cosang = np.clip(uvec @ v_center, -1.0, 1.0)
maxdist_deg = float(np.degrees(np.arccos(cosang)).max())
fov_deg = max(2.0, 2.15 * maxdist_deg)

xsize = int(np.clip(np.ceil(fov_deg * 60.0 / 2.0), 350, 1400))
reso = fov_deg * 60.0 / xsize

# ---------- 3) Project maps ----------
sim_norm = sim / np.mean(sim)

proj_sim = hp.gnomview(
    sim_norm, rot=rot, xsize=xsize, ysize=xsize, reso=reso,
    no_plot=True, return_projected_map=True
)
proj_dr1 = hp.gnomview(
    mask_dr1, rot=rot, xsize=xsize, ysize=xsize, reso=reso,
    no_plot=True, return_projected_map=True
)
proj_tr1 = hp.gnomview(
    mask_tr1, rot=rot, xsize=xsize, ysize=xsize, reso=reso,
    no_plot=True, return_projected_map=True
)
proj_patch = hp.gnomview(
    mask_patch_rot, rot=rot, xsize=xsize, ysize=xsize, reso=reso,
    no_plot=True, return_projected_map=True
)

# ---------- 4) Plot (no color mixing in overlaps) ----------
fig, ax = plt.subplots(figsize=(7.0, 6.2))
extent = (-fov_deg / 2.0, fov_deg / 2.0, -fov_deg / 2.0, fov_deg / 2.0)

# grayscale background normalized to RGB (nonlinear contrast)
base = proj_sim.astype(float)
lo, hi = np.nanpercentile(base, [2, 98])          # percentile stretch
base = np.clip((base - lo) / (hi - lo + 1e-12), 0.0, 1.0)
gamma = 0.6                                        # <1 lifts faint structure, >1 darkens
base = base ** gamma                               # nonlinear contrast
rgb = np.dstack([base, base, base])

# Priority order: first wins in overlap
styles = [
    ("Small", proj_patch, (0.267, 0.467, 0.667)),
    ("Medium",   proj_tr1,   (0.800, 0.733, 0.267)),
    ("Large",   proj_dr1,   (0.933, 0.400, 0.467)),
]
alpha = 0.75

painted = np.zeros(base.shape, dtype=bool)
for _, proj, color in styles:
    m = (proj > 0) & (~painted)
    c = np.array(color)[None, None, :]
    rgb[m] = (1 - alpha) * rgb[m] + alpha * c
    painted |= (proj > 0)

ax.imshow(rgb, origin="lower", extent=extent)

# boundaries for readability
for _, proj, color in styles:
    ax.contour(
        (proj > 0).astype(float),
        levels=[0.5],
        colors=[color],
        linewidths=1.2,
        origin="lower",
        extent=extent
    )

# optional tight crop to occupied area
occ = (proj_dr1 > 0) | (proj_tr1 > 0) | (proj_patch > 0)
yy, xx = np.where(occ)
if xx.size > 0:
    x_min, x_max, y_min, y_max = extent
    dx = (x_max - x_min) / proj_sim.shape[1]
    dy = (y_max - y_min) / proj_sim.shape[0]
    x0 = x_min + xx.min() * dx
    x1 = x_min + (xx.max() + 1) * dx
    y0 = y_min + yy.min() * dy
    y1 = y_min + (yy.max() + 1) * dy
    pad_x = 0.03 * (x1 - x0)
    pad_y = 0.03 * (y1 - y0)
    ax.set_xlim(x0 - pad_x, x1 + pad_x)
    ax.set_ylim(y0 - pad_y, y1 + pad_y)

ax.set_xlabel("Degrees")
ax.set_ylabel("Degrees")
ax.set_title("Gower Street Simulation")
ax.set_aspect("equal")

legend_elements = [
    Patch(facecolor=(0.267, 0.467, 0.667), edgecolor="k", label="Small"),
    Patch(facecolor=(0.800, 0.733, 0.267), edgecolor="k", label="Medium"),
    Patch(facecolor=(0.933, 0.400, 0.467), edgecolor="k", label="Large"),
]
ax.legend(handles=legend_elements, loc="lower right", frameon=True)

fig.tight_layout()
fig.savefig("./plots/masks.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Cls

In [ ]:
## full-sky
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/dummy/"
full_sky_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    full_sky_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax_partial}.fits")
full_sky_cqs = heracles.binned(full_sky_cls, ledges)

# TR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/"
tr1_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    tr1_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax_partial}.fits")
tr1_cqs = heracles.binned(tr1_cls, ledges)

## DR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/"
dr1_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax_partial}.fits")
dr1_cqs = heracles.binned(dr1_cls, ledges)

## PATCH
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/"
patch_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_cls[i] = heracles.read(path+f"/cls/cls_data_{i}_lmax_{lmax_partial}.fits")
patch_cqs = heracles.binned(patch_cls, ledges)

In [ ]:
from dataclasses import replace

def get_cls_mean(cls_dict):
    n_keys = list(cls_dict.keys())
    f_keys = list(cls_dict[n_keys[0]].keys())
    cls_mean = {}
    for f_key in f_keys:
        cl = np.mean([cls_dict[i][f_key] for i in n_keys], axis=0)
        cls_mean[f_key] = replace(cls_dict[i][f_key], array=cl)
    return cls_mean

def get_cls_std(cls_dict):
    n_keys = list(cls_dict.keys())
    f_keys = list(cls_dict[n_keys[0]].keys())
    cls_std = {}
    for f_key in f_keys:
        cl = np.std([cls_dict[i][f_key] for i in n_keys], axis=0)
        cls_std[f_key] = replace(cls_dict[i][f_key], array=cl)
    return cls_std

In [ ]:
## Full Sky means
full_sky_cls_m, full_sky_cls_s = get_cls_mean(full_sky_cls), get_cls_std(full_sky_cls)
full_sky_cqs_m, full_sky_cqs_s = get_cls_mean(full_sky_cqs), get_cls_std(full_sky_cqs)

## TR1
tr1_cls_m, tr1_cls_s = get_cls_mean(tr1_cls), get_cls_std(tr1_cls)
tr1_cqs_m, tr1_cqs_s = get_cls_mean(tr1_cqs), get_cls_std(tr1_cqs)

## PATCH
patch_cls_m, patch_cls_s = get_cls_mean(patch_cls), get_cls_std(patch_cls)
patch_cqs_m, patch_cqs_s = get_cls_mean(patch_cqs), get_cls_std(patch_cqs)

## DR1
dr1_cls_m, dr1_cls_s = get_cls_mean(dr1_cls), get_cls_std(dr1_cls)
dr1_cqs_m, dr1_cqs_s = get_cls_mean(dr1_cqs), get_cls_std(dr1_cqs)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.subplots_adjust(wspace=0.25)

ells = np.arange(2, lmax_partial+1)
mask_plot = {
    "Small mask": {"m": patch_cqs_m, "s": patch_cqs_s, "color": "#4477AA"},
    "Medium mask":   {"m": tr1_cqs_m,   "s": tr1_cqs_s,   "color": "#CCBB44"},
    "Large mask":   {"m": dr1_cqs_m,   "s": dr1_cqs_s,   "color": "#EE6677"},
}

# title, key, component, symlog-linthresh, y-lims
panels = [
    ("PP", ("POS", "POS", 1, 1), None,   1e-8, (-5e-7, 1e-3)),
    ("PE", ("POS", "SHE", 1, 1), (0,),   1e-9, (-1e-4, 1e-9)),
    ("EE", ("SHE", "SHE", 1, 1), (0, 0), 1e-10, (-1e-10, 5e-5)),
    ("BB", ("SHE", "SHE", 1, 1), (1, 1), 1e-10, (-1e-10, 5e-8)),
]

for i, (ax, (title, key, comp, linthresh, ylim)) in enumerate(zip(axes, panels)):
    t = full_sky_cqs_m[key]
    s = full_sky_cqs_s[key]
    t_arr = t.array if hasattr(t, "array") else np.asarray(t)
    s_arr = s.array if hasattr(s, "array") else np.asarray(s)
    if comp is not None:
        t_arr = t_arr[comp]
        s_arr = s_arr[comp]
    #t_arr = t_arr[2:]
    #s_arr = s_arr[2:]

    # only first panel contributes legend labels
    ref_label = "Full sky" if i == 0 else "_nolegend_"
    ax.errorbar(lgrid, lgrid * t_arr, yerr=lgrid * s_arr, c="k", lw=1.2, label=ref_label)

    for name, d in mask_plot.items():
        c_arr= d["m"][key]
        s_arr = d["s"][key]
        if comp is not None:
            c_arr = c_arr[comp]
            s_arr = s_arr[comp]
        #c_arr = c_arr[2:]
        #s_arr = s_arr[2:]

        line_label = name if i == 0 else "_nolegend_"
        ax.errorbar(
            lgrid, lgrid * c_arr, yerr=lgrid * s_arr,
            fmt="-", ms=3, lw=1.0, alpha=0.65,
            color=d["color"], label=line_label
        )

    ax.set_title(title, y=0.85)
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=linthresh, linscale=0.45)
    ax.set_ylim(*ylim)
    #ax.set_xlim(max(7, ell.min()), lmax * 1.05)
    ax.tick_params(axis="both", which="both", direction="in")
    ax.set_xlabel(r"$\ell$")

    if i == 0:
        ax.set_ylabel(r"$\ell C_\ell$")
    else:
        ax.set_ylabel("")  # remove y-labels beyond first panel

axes[0].legend(bbox_to_anchor=(0.5, 1.25), ncols=4, loc="upper left")
fig.savefig("./plots/cls_comp_masks_4x1.pdf", bbox_inches="tight")
plt.show()


## Correlation Function

In [ ]:
## full-sky
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/dummy/"
full_sky_wcls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    full_sky_wcls[i] = heracles.read(path+f"/cls/wcls_data_{i}_lmax_{lmax_partial}.fits")
    
# TR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/"
tr1_wcls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    tr1_wcls[i] = heracles.read(path+f"/cls/wcls_data_{i}_lmax_{lmax_partial}.fits")

## DR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/"
dr1_wcls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_wcls[i] = heracles.read(path+f"/cls/wcls_data_{i}_lmax_{lmax_partial}.fits")

## PATCH
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/"
patch_wcls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_wcls[i] = heracles.read(path+f"/cls/wcls_data_{i}_lmax_{lmax_partial}.fits")

In [ ]:
## Full Sky means
full_sky_wcls_m, full_sky_wcls_s = get_cls_mean(full_sky_wcls), get_cls_std(full_sky_wcls)
## TR1
tr1_wcls_m, tr1_wcls_s = get_cls_mean(tr1_wcls), get_cls_std(tr1_wcls)
## PATCH
patch_wcls_m, patch_wcls_s = get_cls_mean(patch_wcls), get_cls_std(patch_wcls)
## DR1
dr1_wcls_m, dr1_wcls_s = get_cls_mean(dr1_wcls), get_cls_std(dr1_wcls)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.subplots_adjust(wspace=0.25)

xvals2, _ = heracles.transforms._cached_gauss_legendre(lmax_mask+1)
theta = (180/np.pi)*np.arccos(xvals2[::-1])

mask_plot = {
    "Small mask": {"m": patch_wcls_m, "s": patch_wcls_s, "color": "#4477AA"},
    "Medium mask":   {"m": tr1_wcls_m,   "s": tr1_wcls_s,   "color": "#CCBB44"},
    "Large mask":   {"m": dr1_wcls_m,   "s": dr1_wcls_s,   "color": "#EE6677"},
}

# title, key, component, symlog-linthresh, y-lims
panels = [
    ("PP", ("POS", "POS", 1, 1), None,   1e-7, (-5e-7, 1e-1)),
    ("PE", ("POS", "SHE", 1, 1), (0,),   1e-8, (-1e-3, 1e-9)),
    ("EE", ("SHE", "SHE", 1, 1), (0, 0), 1e-9, (-1e-10, 5e-4)),
    ("BB", ("SHE", "SHE", 1, 1), (1, 1), 1e-10, (-1e-10, 5e-4)),
]

for i, (ax, (title, key, comp, linthresh, ylim)) in enumerate(zip(axes, panels)):
    t = full_sky_wcls_m[key]
    s = full_sky_wcls_s[key]
    t_arr = t.array if hasattr(t, "array") else np.asarray(t)
    s_arr = s.array if hasattr(s, "array") else np.asarray(s)
    if comp is not None:
        t_arr = t_arr[comp]
        s_arr = s_arr[comp]
    t_arr = t_arr[::-1]
    s_arr = s_arr[::-1]

    # only first panel contributes legend labels
    ref_label = "Full sky" if i == 0 else "_nolegend_"
    ax.plot(theta, t_arr, c="k", lw=1.2, label=ref_label)

    for name, d in mask_plot.items():
        c_arr= d["m"][key]
        s_arr = d["s"][key]
        if comp is not None:
            c_arr = c_arr[comp]
            s_arr = s_arr[comp]
        c_arr = c_arr[::-1]
        s_arr = s_arr[::-1]
        line_label = name if i == 0 else "_nolegend_"
        ax.plot(
            theta, c_arr, "-", ms=3, lw=1.0, alpha=0.65,
            color=d["color"], label=line_label
        )

    ax.set_title(title, y=0.85)
    ax.set_yscale("symlog", linthresh=linthresh, linscale=0.45)
    ax.set_ylim(*ylim)
    ax.set_xscale("log")
    #ax.set_xlim(max(7, ell.min()), lmax * 1.05)
    ax.tick_params(axis="both", which="both", direction="in")
    ax.set_xlabel(r"$\theta$")

    if i == 0:
        ax.set_ylabel(r"$\xi^{ff'}(\theta)$")
    else:
        ax.set_ylabel("")  # remove y-labels beyond first panel

axes[0].legend(bbox_to_anchor=(0.5, 1.25), ncols=4, loc="upper left")
fig.savefig("./plots/wcls_comp_masks.pdf", bbox_inches="tight")
plt.show()


In [ ]:
patch_mls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/cls/cls_mask_lmax_{lmax_mask}.fits")
tr1_mls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/cls/cls_mask_lmax_{lmax_mask}.fits")
dr1_mls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/cls/cls_mask_lmax_{lmax_mask}.fits")

patch_wmls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/cls/wcls_mask_lmax_{lmax_mask}.fits")
tr1_wmls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/cls/wcls_mask_lmax_{lmax_mask}.fits")
dr1_wmls = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/cls/wcls_mask_lmax_{lmax_mask}.fits")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

def norm_last(arr, eps=1e-30):
    return arr if np.abs(arr[-1]) < eps else arr / arr[-1]

rows = [
    dict(name="Small",  color="#4477AA", corr=patch_wmls["WHT", "WHT", 1, 1].array,),
    dict(name="Medium", color="#CCBB44", corr=tr1_wmls["WHT", "WHT", 1, 1].array,),
    dict(name="Large",  color="#EE6677", corr=dr1_wmls["WHT", "WHT", 1, 1].array,),
]

c = 0  # identical columns

fig, ax = plt.subplots(figsize=(7.2, 4.8))

# Set limits BEFORE shading so axvspan uses correct bounds
ax.set_xscale("log")
ax.set_xlim(0, 150)
ax.set_ylim(-1e-1, 1.2e0)

for i, row in enumerate(rows):
    f_unc = gaussian_filter1d(norm_last(row["corr"]), sigma=2)
    yvals = f_unc[::-1]
    
    ax.plot(
        theta, yvals,
        color=row["color"], alpha=0.95, lw=2,
        label=f'{row["name"]}'
    )
    
    theta_star_i = np.abs(yvals - rcond).argmin()
    theta_star = theta[theta_star_i]

    
    print(row["name"], theta_star, 3*int(180/theta_star))
    
    # Shade region to the right of theta_star
    ax.axvspan(
        theta_star, ax.get_xlim()[1],
        color=row["color"], alpha=0.15
    )

    # Add LaTeX-formatted annotation
    y_text = 1.05 - 3 * 0.08  # stagger labels vertically
    ax.text(
        theta_star * 1.05, y_text,
        rf"$\theta^{{\star}} = {theta_star:.1f}^\circ$",
        color=row["color"],
        fontweight="bold",
        fontsize=12,
        ha='left',
        va='center'
    )

#ax.set_yscale("symlog", linthresh=1e-3)
ax.set_xlabel(r"$\theta [\rm{deg}]$")
ax.set_ylabel(r"$\xi^{ww'}(\theta)$")
ax.legend(ncols=3, frameon=False, loc="upper center")

fig.savefig(f"./plots/wm_comparison_rcond_{rcond}.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
from heracles.fields import Positions, Shears, Visibility, Weights
from heracles.healpy import HealpixMapper

mask_mapper = HealpixMapper(nside=nside, lmax=lmax_partial, deconvolve=False)
mask_fields = {
    "POS": Positions(mask_mapper, mask="VIS"),
    "SHE": Shears(mask_mapper, mask="WHT"),
    "VIS": Visibility(mask_mapper),
    "WHT": Weights(mask_mapper),
}

def corr_wmls(wmls, rcond=0.001):
    wmls_corr = {}
    for m_key in list(wmls.keys()):
        wml = np.copy(wmls[m_key].array)
        wml = wml * heracles.unmixing.logistic(np.log10(abs(wml)), x0=np.log10(rcond*np.max(wml)), k=100)
        wmls_corr[m_key] = replace(wmls[m_key], array=wml)
    return wmls_corr

patch_corr_wmls = corr_wmls(patch_wmls, rcond=rcond)
dr1_corr_wmls = corr_wmls(dr1_wmls, rcond=rcond)
tr1_corr_wmls = corr_wmls(tr1_wmls, rcond=rcond)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

xvals2, _ = heracles.transforms._cached_gauss_legendre(lmax_mask+1)

def norm_last(arr, eps=1e-30):
    return arr if np.abs(arr[-1]) < eps else arr / arr[-1]

rows = [
    dict(name="Small",  color="#4477AA", corr=1/patch_corr_wmls["WHT", "WHT", 1, 1].array,),
    dict(name="Medium", color="#CCBB44", corr=1/tr1_corr_wmls["WHT", "WHT", 1, 1].array,),
    dict(name="Large",  color="#EE6677", corr=1/dr1_corr_wmls["WHT", "WHT", 1, 1].array,),
]

c = 0  # identical columns

fig, ax = plt.subplots(figsize=(7.2, 4.8))

# Set limits BEFORE shading so axvspan uses correct bounds
ax.set_xscale("log")
ax.set_xlim(0, 150)
ax.set_ylim(-1e-1, 1e3)

for i, row in enumerate(rows):
    f_unc = gaussian_filter1d(norm_last(row["corr"]), sigma=2)
    theta = (180/np.pi)*np.arccos(xvals2[::-1])
    
    yvals = f_unc[::-1]
    
    ax.plot(
        theta, yvals,
        color=row["color"], alpha=0.95, lw=2,
        label=f'{row["name"]}'
    )
    
    theta_star_i = np.abs(yvals - rcond).argmin()
    theta_star = theta[theta_star_i]

    
    print(row["name"], theta_star)
    
    # Shade region to the right of theta_star
    ax.axvspan(
        theta_star, ax.get_xlim()[1],
        color=row["color"], alpha=0.15
    )

    # Add LaTeX-formatted annotation
    y_text = 1.05 - 3 * 0.08  # stagger labels vertically
    ax.text(
        theta_star * 1.05, y_text,
        rf"$\theta^{{\star}} = {theta_star:.1f}^\circ$",
        color=row["color"],
        fontweight="bold",
        fontsize=12,
        ha='left',
        va='center'
    )

ax.set_yscale("symlog")
ax.set_xlabel(r"$\theta [\rm{deg}]$")
ax.set_ylabel(r"$(\xi^{ww'}(\theta))^{-1}$")
ax.legend(ncols=3, frameon=False, loc="upper center")

fig.savefig(f"./plots/inv_wm_comparison_rcond_{rcond}.pdf", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
_patch_nu_wcls = heracles.unmixing._naturalspice(patch_wcls_m, patch_corr_wmls, mask_fields)
_patch_nu_cls  = heracles.corr2cl(_patch_nu_wcls)
_dr1_nu_wcls = heracles.unmixing._naturalspice(dr1_wcls_m, dr1_corr_wmls, mask_fields)
_dr1_nu_cls  = heracles.corr2cl(_dr1_nu_wcls)
_tr1_nu_wcls = heracles.unmixing._naturalspice(tr1_wcls_m, tr1_corr_wmls, mask_fields)
_tr1_nu_cls  = heracles.corr2cl(_tr1_nu_wcls)

_patch_nu_cqs = heracles.binned(_patch_nu_cls, ledges)
_dr1_nu_cqs = heracles.binned(_dr1_nu_cls, ledges)
_tr1_nu_cqs = heracles.binned(_tr1_nu_cls, ledges)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
fig.subplots_adjust(wspace=0.25)

xvals2, _ = heracles.transforms._cached_gauss_legendre(lmax_mask+1)
theta = (180/np.pi)*np.arccos(xvals2[::-1])

mask_plot = {
    "Small mask":  {"m": _patch_nu_wcls, "color": "#4477AA"},
    "Medium mask": {"m": _tr1_nu_wcls,   "color": "#CCBB44"},
    "Large mask":  {"m": _dr1_nu_wcls,   "color": "#EE6677"},
}

# title, key, component, symlog-linthresh, y-lims
panels = [
    ("PP", ("POS", "POS", 1, 1), None,   1e-7, (-5e-7, 1e-1)),
    ("PE", ("POS", "SHE", 1, 1), (0,),   1e-8, (-1e-3, 1e-9)),
    ("EE", ("SHE", "SHE", 1, 1), (0, 0), 1e-9, (-1e-10, 5e-4)),
    ("BB", ("SHE", "SHE", 1, 1), (1, 1), 1e-10, (-1e-10, 5e-4)),
]

for i, (ax, (title, key, comp, linthresh, ylim)) in enumerate(zip(axes, panels)):
    t = full_sky_wcls_m[key]
    t_arr = t.array if hasattr(t, "array") else np.asarray(t)
    if comp is not None:
        t_arr = t_arr[comp]
    t_arr = t_arr[::-1]

    # only first panel contributes legend labels
    ref_label = "Full sky" if i == 0 else "_nolegend_"
    ax.plot(theta, t_arr, c="k", lw=1.2, label=ref_label)

    for name, d in mask_plot.items():
        c_arr= d["m"][key]
        if comp is not None:
            c_arr = c_arr[comp]
        c_arr = c_arr[::-1]
        line_label = name if i == 0 else "_nolegend_"
        ax.plot(
            theta, c_arr, "-", ms=3, lw=1.0, alpha=0.65,
            color=d["color"], label=line_label
        )

    ax.set_title(title, y=0.85)
    ax.set_yscale("symlog", linthresh=linthresh, linscale=0.45)
    ax.set_ylim(*ylim)
    ax.set_xscale("log")
    #ax.set_xlim(max(7, ell.min()), lmax * 1.05)
    ax.tick_params(axis="both", which="both", direction="in")
    ax.set_xlabel(r"$\theta$")

    if i == 0:
        ax.set_ylabel(r"$\xi^{ff'}(\theta)$")
    else:
        ax.set_ylabel("")  # remove y-labels beyond first panel

axes[0].legend(bbox_to_anchor=(0.5, 1.25), ncols=4, loc="upper left")
fig.savefig("./plots/wcls_comp_masks.pdf", bbox_inches="tight")
plt.show()


## MixMats

In [ ]:
tr1_mixmat = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/mixmat_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")
dr1_mixmat = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/mixmat_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")
patch_mixmat = heracles.read(f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/mixmat_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")

## Deconvolution

In [ ]:

## patch
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/patch/"
patch_nu_cls = {}
patch_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_rcond_{rcond}_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")
    patch_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax_partial}.fits")

patch_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    patch_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmin_{lmin}_lmax_{lmax_partial}.fits")
patch_nu_cqs = heracles.binned(patch_nu_cls, ledges)
patch_pols_cqs = heracles.binned(patch_pols_cls, ledges)
patch_nmt_cqs = heracles.binned(patch_nmt_cqs, ledges)

# TR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/tr1/"
tr1_nu_cls = {}
tr1_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    tr1_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_rcond_{rcond}_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")
    tr1_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax_partial}.fits")

tr1_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    tr1_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmin_{lmin}_lmax_{lmax_partial}.fits")
tr1_nu_cqs = heracles.binned(tr1_nu_cls, ledges)
tr1_pols_cqs = heracles.binned(tr1_pols_cls, ledges)
tr1_nmt_cqs = heracles.binned(tr1_nmt_cqs, ledges)

## DR1
path = f"/pscratch/sd/j/jaimerz/{mode}_sims/dr1/"
dr1_nu_cls = {}
dr1_pols_cls = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_nu_cls[i] = heracles.read(path+f"/cls_nu/cls_data_nu_{i}_rcond_{rcond}_l1max_{lmax_partial}_l2max_{lmax_mask}.fits")
    dr1_pols_cls[i] = heracles.read(path+f"/cls_pols/cls_data_pols_{i}_lmax_{lmax_partial}.fits")

dr1_nmt_cqs = {}
for i in range(1, n+1):
    print(f"Loading sim {i}", end='\r')
    dr1_nmt_cqs[i] = heracles.read(path+f"/cls_nmt/cqs_data_nmt_np_{i}_lmin_{lmin}_lmax_{lmax_partial}.fits")

dr1_nu_cqs = heracles.binned(dr1_nu_cls, ledges)
dr1_pols_cqs = heracles.binned(dr1_pols_cls, ledges)
dr1_nmt_cqs = heracles.binned(dr1_nmt_cqs, ledges)

In [ ]:
## PATCH
patch_nu_cqs_m, patch_nu_cqs_s = get_cls_mean(patch_nu_cqs), get_cls_std(patch_nu_cqs)
patch_pols_cqs_m, patch_pols_cqs_s = get_cls_mean(tr1_pols_cqs), get_cls_std(patch_pols_cqs)
patch_nmt_cqs_m, patch_nmt_cqs_s = get_cls_mean(patch_nmt_cqs), get_cls_std(patch_nmt_cqs)

## TR1
tr1_nu_cqs_m, tr1_nu_cqs_s = get_cls_mean(tr1_nu_cqs), get_cls_std(tr1_nu_cqs)
tr1_pols_cqs_m, tr1_pols_cqs_s = get_cls_mean(tr1_pols_cqs), get_cls_std(tr1_pols_cqs)
tr1_nmt_cqs_m, tr1_nmt_cqs_s = get_cls_mean(tr1_nmt_cqs), get_cls_std(tr1_nmt_cqs)

## DR1 
dr1_nu_cqs_m, dr1_nu_cqs_s = get_cls_mean(dr1_nu_cqs), get_cls_std(dr1_nu_cqs)
dr1_pols_cqs_m, dr1_pols_cqs_s = get_cls_mean(dr1_pols_cqs), get_cls_std(dr1_pols_cqs)
dr1_nmt_cqs_m, dr1_nmt_cqs_s = get_cls_mean(dr1_nmt_cqs), get_cls_std(dr1_nmt_cqs)

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde

rows = [
    ("Small",  patch_nu_cqs_s, patch_nmt_cqs_s, patch_pols_cqs_s, patch_cqs_s, fsky_patch, lgrid>500),
    ("Medium", tr1_nu_cqs_s,   tr1_nmt_cqs_s,   tr1_pols_cqs_s, tr1_cqs_s, fsky_tr1, lgrid>300),
    ("Large",  dr1_nu_cqs_s,   dr1_nmt_cqs_s,   dr1_pols_cqs_s, dr1_cqs_s, fsky_dr1, lgrid>20),
]
cols = [
    ("PP", ("POS", "POS", 1, 1), lambda a: a[:]),
    ("PE", ("POS", "SHE", 1, 1), lambda a: a[0, :]),
    ("PB", ("POS", "SHE", 1, 1), lambda a: a[1, :]),
    ("EE", ("SHE", "SHE", 1, 1), lambda a: a[0, 0, :]),
    ("EB", ("SHE", "SHE", 1, 1), lambda a: a[0, 1, :]),
    ("BB", ("SHE", "SHE", 1, 1), lambda a: a[1, 1, :]),
]
def kde_mode(x, n_grid=512):
    x = np.asarray(x, dtype=float).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    if x.size < 3 or np.allclose(x, x[0]):
        return float(np.median(x))
    kde = gaussian_kde(x)
    lo, hi = np.min(x), np.max(x)
    if lo == hi:
        return float(lo)
    grid = np.linspace(lo, hi, n_grid)
    return float(grid[np.argmax(kde(grid))])


def _central(vals, central):
    vals = np.asarray(vals, dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None
    if central == "mean":
        return float(np.mean(vals))
    elif central == "median":
        return float(np.median(vals))
    elif central == "mode":
        return kde_mode(vals)
    raise ValueError(f"unknown central='{central}'")


def _ratio_vals(nu, nmt, key, extract, sel):
    """Per-multipole symmetric fractional difference between the two methods:
    (sigma^NaturalSpice - sigma^NaMaster) / (0.5 * (sigma^NaturalSpice + sigma^NaMaster))."""
    s_nu = extract(nu[key])
    s_nmt = extract(nmt[key])
    return ((s_nu - s_nmt) / (0.5 * (s_nu + s_nmt)))[sel]


def make_latex_table(
    rows,
    cols,
    central="mean",          # "mean", "median", or "mode"
    fmt="{:.2f}",
    quantity=(
        r"(\sigma_\ell^{\tt NaturalSpice} - \sigma_\ell^{\tt NaMaster}) / "
        r"\tfrac{1}{2}(\sigma_\ell^{\tt NaturalSpice} + \sigma_\ell^{\tt NaMaster})"
    ),
    caption=(
        "Symmetric fractional difference between the \\texttt{NaturalSpice} and "
        "\\texttt{NaMaster} angular power spectrum uncertainties, averaged over the "
        "retained multipole range. Different rows show different masks while "
        "different columns show different auto- and cross-correlations."
    ),
    label="tab:sigma_ratio",
):
    ncol = len(cols)
    colspec = "l" + "c" * ncol
    field_labels = [lab for (lab, _key, _ex) in cols]

    out = []
    out.append(r"\begin{table*}")
    out.append(r"\centering")
    out.append(r"\renewcommand{\arraystretch}{1.4}   % <-- adjust this number to taste")
    out.append(r"\begin{tabular}{" + colspec + "}")
    out.append(r"\hline")
    out.append(r"\multicolumn{" + str(ncol + 1) + r"}{c}{$" + quantity + r"$} \\")
    out.append(r"\hline")
    out.append("Mask & " + " & ".join(field_labels) + r" \\")
    out.append(r"\hline")

    for row in rows:
        name, nu, nmt, sel = row[0], row[1], row[2], row[6]
        mask_name = f"{name} mask"

        cells = []
        for (lab, key, extract) in cols:
            vals = _ratio_vals(nu, nmt, key, extract, sel)
            c = _central(vals, central)
            cells.append(r"$-$" if c is None else "$" + fmt.format(c) + "$")

        out.append(f"{mask_name} & " + " & ".join(cells) + r" \\")

    out.append(r"\hline")
    out.append(r"\end{tabular}")
    out.append(r"\caption{" + caption + "}")
    out.append(r"\label{" + label + "}")
    out.append(r"\end{table*}")
    return "\n".join(out)


if __name__ == "__main__":
    tex = make_latex_table(rows, cols, central="mean")
    print(tex)
    with open("./plots/sigma_ratio_table.tex", "w") as f:
        f.write(tex + "\n")

In [ ]:
# import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# -----------------------------
# Config
# -----------------------------
_theory_cls = full_sky_cqs_m
xlim = (lgrid[0], lgrid[-1])  # <-- adjust this to limit the x-range

# low-ell cut shaded in the residual panels (per mask), as in the first figure
region_lmin = {
    "Small mask": 120,
    "Medium mask": 24,
    "Large mask": 10,
}

plotting_elements = {
    "Small mask": {
        "nmt_c_m": patch_nmt_cqs_m,
        "nu_c_m": patch_nu_cqs_m,
        "pols_c_m": patch_pols_cqs_m,
        "nmt_c_s": patch_nmt_cqs_s,
        "nu_c_s": patch_nu_cqs_s,
        "pols_c_s": patch_pols_cqs_s,
    },
    "Medium mask": {
        "nmt_c_m": tr1_nmt_cqs_m,
        "nu_c_m": tr1_nu_cqs_m,
        "pols_c_m": tr1_pols_cqs_m,
        "nmt_c_s": tr1_nmt_cqs_s,
        "nu_c_s": tr1_nu_cqs_s,
        "pols_c_s": tr1_pols_cqs_s,
    },
    "Large mask": {
        "nmt_c_m": dr1_nmt_cqs_m,
        "nu_c_m": dr1_nu_cqs_m,
        "pols_c_m": dr1_pols_cqs_m,
        "nmt_c_s": dr1_nmt_cqs_s,
        "nu_c_s": dr1_nu_cqs_s,
        "pols_c_s": dr1_pols_cqs_s,
    },
}

method_order = ["NaturalSpice", "NaMaster", "PolSpice"]
method_keys_m = {
    "NaturalSpice": "nu_c_m",
    "NaMaster": "nmt_c_m",
    "PolSpice": "pols_c_m",
}
method_keys_s = {
    "NaturalSpice": "nu_c_s",
    "NaMaster": "nmt_c_s",
    "PolSpice": "pols_c_s",
}

# color encodes the mask
region_order = ["Small mask", "Medium mask", "Large mask"]
region_colors = {
    "Small mask": "#4477AA",
    "Medium mask": "#CCBB44",
    "Large mask": "#EE6677",
}

spec_cfg = [
    {
        "title": r"$\ell  \hat{C}_\ell^{\rm PP}$",
        "key": ("POS", "POS", 1, 1),
        "extract": lambda a: a[:],
        "ylim": (-2e-4, 5e-4),
        "linthresh": 1e-4,
    },
    {
        "title": r"$\ell  \hat{C}_\ell^{\rm PE}$",
        "key": ("POS", "SHE", 1, 1),
        "extract": lambda a: a[0, :],
        "ylim": (-5e-5, 5e-5),
        "linthresh": 1e-5,
    },
    {
        "title": r"$\ell  \hat{C}_\ell^{\rm EE}$",
        "key": ("SHE", "SHE", 1, 1),
        "extract": lambda a: a[0, 0, :],
        "ylim": (-5e-6, 5e-5),
        "linthresh": 1e-6,
    },
    {
        "title": r"$\ell  \hat{C}_\ell^{\rm BB}$",
        "key": ("SHE", "SHE", 1, 1),
        "extract": lambda a: a[1, 1, :],
        "ylim": (-1e-5, 1e-5),
        "linthresh": 1e-8,
    },
]

# 3 methods (row groups), each = 1 main + one residual row per mask
n_methods = len(method_order)
rows_per_method = 1 + len(region_order)
nrows = n_methods * rows_per_method
# main row tall, residual rows short
height_ratios = ([4] + [1.0] * len(region_order)) * n_methods

fig, ax = plt.subplots(
    nrows, 4, figsize=(14, 12),
    gridspec_kw={"height_ratios": height_ratios},
)
fig.subplots_adjust(left=0.06, bottom=0.05, right=0.995, top=0.94, wspace=0.22, hspace=0.03)

for imeth, m in enumerate(method_order):
    row0 = imeth * rows_per_method  # main row for this method
    is_last_method = (imeth == n_methods - 1)

    for icol, cfg in enumerate(spec_cfg):
        key = cfg["key"]
        extract = cfg["extract"]
        t = extract(_theory_cls[key])

        # -------- Main panel: all masks overlaid + theory (fill_between)
        a_main = ax[row0, icol]
        for region in region_order:
            pe = plotting_elements[region]
            c = extract(pe[method_keys_m[m]][key])
            e = extract(pe[method_keys_s[m]][key])
            y = lgrid * c
            dy = lgrid * e
            # 1-sigma band
            a_main.fill_between(
                lgrid, y - dy, y + dy,
                color=region_colors[region], alpha=0.2,
                linewidth=0, zorder=2.5,
            )
            # boundary lines (upper + lower edges of band)
            a_main.plot(
                lgrid, y - dy,
                color=region_colors[region], lw=0.9, alpha=0.9,
                zorder=3.0, label=region,
            )
            a_main.plot(
                lgrid, y + dy,
                color=region_colors[region], lw=0.9, alpha=0.9,
                zorder=3.0,
            )
        a_main.plot(lgrid, lgrid * t, c="k", lw=1.0, zorder=4.0)
        a_main.set_xscale("log")
        a_main.set_xlim(xlim)
        a_main.set_yscale("symlog", linthresh=cfg["linthresh"], linscale=0.45)
        a_main.set_ylim(cfg["ylim"])
        a_main.tick_params(axis="both", which="both", direction="in")
        a_main.tick_params(labelbottom=False)
        if imeth == 0:
            a_main.set_title(cfg["title"], y=0.79)
        if icol == 0:
            a_main.set_ylabel(f"{m}")

        # -------- Residual subpanels (one mask per row)
        for ireg, region in enumerate(region_order):
            a_res = ax[row0 + 1 + ireg, icol]
            pe = plotting_elements[region]
            c = extract(pe[method_keys_m[m]][key])
            e = extract(pe[method_keys_s[m]][key])
            a_res.errorbar(
                lgrid, (c - t) / e, yerr=np.abs(e / e), fmt=".",
                lw=1.0, alpha=0.6, zorder=2.0, color=region_colors[region],
            )
            a_res.axvspan(xlim[0], region_lmin[region], color="k", alpha=0.1, linewidth=0, zorder=0)
            a_res.axhline(0.0, c="k", lw=0.8, zorder=-1)
            a_res.set_xscale("log")
            a_res.set_xlim(xlim)
            a_res.set_yticks([-1, 0, 1])
            a_res.set_ylim(-1.8, 1.8)
            a_res.tick_params(axis="both", which="both", direction="in")
            # only the very last row of the figure gets x-ticks/label
            is_last_row = is_last_method and (ireg == len(region_order) - 1)
            if is_last_row:
                a_res.set_xlabel(r"$\ell$")
            else:
                a_res.tick_params(labelbottom=False)

# Shared legend (theory first, then masks)
legend_handles = [Line2D([0], [0], color="k", lw=1.0, label="Full Sky")]
legend_handles += [
    Line2D([0], [0], color=region_colors[r], lw=1.2, label=r)
    for r in region_order
]
fig.legend(
    handles=legend_handles,
    loc="upper center", ncol=4, frameon=False,
    bbox_to_anchor=(0.5, 0.99),
)

fig.savefig(f"./plots/cqs_comp_2_rcond_{rcond}.pdf", bbox_inches="tight")
plt.show()

In [ ]:
## full_sky covariances
full_sky_ensemble_cov = dices.jackknife_covariance(full_sky_cqs, nd=0)

## patch covariances
patch_ensemble_cov = dices.jackknife_covariance(patch_cqs, nd=0)
patch_nu_ensemble_cov = dices.jackknife_covariance(patch_nu_cqs, nd=0)
patch_nmt_ensemble_cov = dices.jackknife_covariance(patch_nmt_cqs, nd=0)
patch_pols_ensemble_cov = dices.jackknife_covariance(patch_pols_cqs, nd=0)

## half_sky covariances
tr1_ensemble_cov = dices.jackknife_covariance(tr1_cqs, nd=0)
tr1_nu_ensemble_cov = dices.jackknife_covariance(tr1_nu_cqs, nd=0)
tr1_nmt_ensemble_cov = dices.jackknife_covariance(tr1_nmt_cqs, nd=0)
tr1_pols_ensemble_cov = dices.jackknife_covariance(tr1_pols_cqs, nd=0)

## Planck covariances
dr1_ensemble_cov = dices.jackknife_covariance(dr1_cqs, nd=0)
dr1_nu_ensemble_cov = dices.jackknife_covariance(dr1_nu_cqs, nd=0)
dr1_nmt_ensemble_cov = dices.jackknife_covariance(dr1_nmt_cqs, nd=0)
dr1_pols_ensemble_cov = dices.jackknife_covariance(dr1_pols_cqs, nd=0)

In [ ]:
full_sky_flat_ensemble_cov = dices.flatten(full_sky_ensemble_cov)
full_sky_flat_ensemble_corr = full_sky_flat_ensemble_cov / np.sqrt(
    np.diag(full_sky_flat_ensemble_cov)[:, None] * np.diag(full_sky_flat_ensemble_cov)[None, :]
)

patch_flat_ensemble_cov = dices.flatten(patch_ensemble_cov)
patch_flat_nu_ensemble_cov = dices.flatten(patch_nu_ensemble_cov)
patch_flat_pols_ensemble_cov = dices.flatten(patch_pols_ensemble_cov)
patch_flat_nmt_ensemble_cov = dices.flatten(patch_nmt_ensemble_cov)

patch_flat_ensemble_corr = patch_flat_ensemble_cov / np.sqrt(
    np.diag(patch_flat_ensemble_cov)[:, None] * np.diag(patch_flat_ensemble_cov)[None, :]
)
patch_flat_nu_ensemble_corr = patch_flat_nu_ensemble_cov / np.sqrt(
    np.diag(patch_flat_nu_ensemble_cov)[:, None] * np.diag(patch_flat_nu_ensemble_cov)[None, :]
)
patch_flat_pols_ensemble_corr = patch_flat_pols_ensemble_cov / np.sqrt(
    np.diag(patch_flat_pols_ensemble_cov)[:, None] * np.diag(patch_flat_pols_ensemble_cov)[None, :]
)
patch_flat_nmt_ensemble_corr = patch_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(patch_flat_nmt_ensemble_cov)[:, None] * np.diag(patch_flat_nmt_ensemble_cov)[None, :]
)

tr1_flat_ensemble_cov = dices.flatten(tr1_ensemble_cov)
tr1_flat_nu_ensemble_cov = dices.flatten(tr1_nu_ensemble_cov)
tr1_flat_pols_ensemble_cov = dices.flatten(tr1_pols_ensemble_cov)
tr1_flat_nmt_ensemble_cov = dices.flatten(tr1_nmt_ensemble_cov)

tr1_flat_ensemble_corr = tr1_flat_ensemble_cov / np.sqrt(
    np.diag(tr1_flat_ensemble_cov)[:, None] * np.diag(tr1_flat_ensemble_cov)[None, :]
)
tr1_flat_nu_ensemble_corr = tr1_flat_nu_ensemble_cov / np.sqrt(
    np.diag(tr1_flat_nu_ensemble_cov)[:, None] * np.diag(tr1_flat_nu_ensemble_cov)[None, :]
)
tr1_flat_pols_ensemble_corr = tr1_flat_pols_ensemble_cov / np.sqrt(
    np.diag(tr1_flat_pols_ensemble_cov)[:, None] * np.diag(tr1_flat_pols_ensemble_cov)[None, :]
)
tr1_flat_nmt_ensemble_corr = tr1_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(tr1_flat_nmt_ensemble_cov)[:, None] * np.diag(tr1_flat_nmt_ensemble_cov)[None, :]
)


dr1_flat_ensemble_cov = dices.flatten(dr1_ensemble_cov)
dr1_flat_nu_ensemble_cov = dices.flatten(dr1_nu_ensemble_cov)
dr1_flat_pols_ensemble_cov = dices.flatten(dr1_pols_ensemble_cov)
dr1_flat_nmt_ensemble_cov = dices.flatten(dr1_nmt_ensemble_cov)

dr1_flat_ensemble_corr = dr1_flat_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_ensemble_cov)[:, None] * np.diag(dr1_flat_ensemble_cov)[None, :]
)
dr1_flat_nu_ensemble_corr = dr1_flat_nu_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_nu_ensemble_cov)[:, None] * np.diag(dr1_flat_nu_ensemble_cov)[None, :]
)
dr1_flat_pols_ensemble_corr = dr1_flat_pols_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_pols_ensemble_cov)[:, None] * np.diag(dr1_flat_pols_ensemble_cov)[None, :]
)
dr1_flat_nmt_ensemble_corr = dr1_flat_nmt_ensemble_cov / np.sqrt(
    np.diag(dr1_flat_nmt_ensemble_cov)[:, None] * np.diag(dr1_flat_nmt_ensemble_cov)[None, :]
)

In [ ]:
def cholesky_or_diagonalize(A, rtol=1e-10):
    """
    Compute a matrix square root S such that A ≈ S @ S.T

    Uses Cholesky if possible, otherwise eigen-decomposition.

    Parameters
    ----------
    A : (n, n) ndarray
        Symmetric matrix
    tol : float
        Threshold for small negative eigenvalues

    Returns
    -------
    S : (n, n) ndarray
        Matrix square root
    """
    try:
        # Fast path: A is symmetric positive definite
        return np.linalg.cholesky(A)

    except np.linalg.LinAlgError:
        # Fallback: symmetric eigen-decomposition
        eigvals, eigvecs = np.linalg.eigh(A)

        # Clamp small negatives due to numerical error
        tol = rtol*np.max(eigvals)
        eigvals[eigvals < tol] = 0.0

        # Build square root: S = Q sqrt(D)
        sqrt_eigvals = np.sqrt(eigvals)
        S = eigvecs @ np.diag(sqrt_eigvals)

        return S

In [ ]:
def combine_matrices(A, B):
    """
    Combines two square matrices:
    - Lower triangle (including diagonal) from A
    - Upper triangle from B
    """
    # Make sure A and B are numpy arrays
    A = np.array(A)
    B = np.array(B)

    # Create an empty matrix
    C = np.zeros_like(A)

    # Use masks for lower and upper triangles
    lower_mask = np.tri(A.shape[0], dtype=bool)  # i >= j
    upper_mask = ~lower_mask                     # i < j

    # Assign values
    C[lower_mask] = A[lower_mask]
    C[upper_mask] = B[upper_mask]

    return C

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(9, 9))
plt.subplots_adjust(hspace=0.0, wspace=0.1)

# ---- Block tick setup ----
labels = ["PP", "PE", "PB", "EE", "EB", "BE", "BB"]
n_bins = 30
positions = [i * n_bins + n_bins / 2 for i in range(len(labels))]

def format_ax(ax, show_xticks=False, show_yticks=False):
    if show_xticks:
        ax.set_xticks(positions)
        ax.set_xticklabels(labels, rotation=45)
    else:
        ax.set_xticks([])

    if show_yticks:
        ax.set_yticks(positions)
        ax.set_yticklabels(labels)
    else:
        ax.set_yticks([])

    # Optional: draw block boundaries
    for k in range(1, len(labels)):
        ax.axhline(k * n_bins - 0.5, color='black', linewidth=0.5)
        ax.axvline(k * n_bins - 0.5, color='black', linewidth=0.5)


# ===================== ROW 1 (Patch) =====================
comb_corr = combine_matrices(full_sky_flat_ensemble_corr, patch_flat_nu_ensemble_corr)
im = axes[0, 0].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0, 0].set_ylabel("Small mask")
axes[0, 0].set_title("NaturalSpice")
format_ax(axes[0, 0], show_yticks=True)

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, patch_flat_nmt_ensemble_corr)
axes[0, 1].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0, 1].set_title("NaMaster")
format_ax(axes[0, 1])

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, patch_flat_pols_ensemble_corr)
axes[0, 2].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[0, 2].set_title("PolSpice")
format_ax(axes[0, 2])


# ===================== ROW 2 (TR1) =====================
comb_corr = combine_matrices(full_sky_flat_ensemble_corr, tr1_flat_nu_ensemble_corr)
axes[1, 0].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[1, 0].set_ylabel("Medium mask")
format_ax(axes[1, 0], show_yticks=True)

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, tr1_flat_nmt_ensemble_corr)
axes[1, 1].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
format_ax(axes[1, 1])

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, tr1_flat_pols_ensemble_corr)
axes[1, 2].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
format_ax(axes[1, 2])


# ===================== ROW 3 (DR1) =====================
comb_corr = combine_matrices(full_sky_flat_ensemble_corr, dr1_flat_nu_ensemble_corr)
axes[2, 0].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
axes[2, 0].set_ylabel("Large mask")
format_ax(axes[2, 0], show_xticks=True, show_yticks=True)

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, dr1_flat_nmt_ensemble_corr)
axes[2, 1].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
format_ax(axes[2, 1], show_xticks=True)

comb_corr = combine_matrices(full_sky_flat_ensemble_corr, dr1_flat_pols_ensemble_corr)
axes[2, 2].imshow(comb_corr, cmap="RdBu_r", vmin=-1, vmax=1)
format_ax(axes[2, 2], show_xticks=True)


# ===================== COLORBAR =====================
cbar = fig.colorbar(im, ax=axes, location="right", fraction=0.03, pad=0.02)
cbar.set_label("Correlation")
cbar.set_ticks([-1, -0.5, 0, 0.5, 1])

plt.show()
fig.savefig(f"./plots/corr_rcond_{rcond}_comp.pdf", bbox_inches="tight")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ref_corr = full_sky_flat_ensemble_corr
eig_ref = np.sort(np.linalg.eigvalsh(ref_corr))[::-1]
eps = 1e-12
den = np.where(np.abs(eig_ref) < eps, np.nan, eig_ref)

groups = {
    "Small mask": {
        "NaturalSpice": patch_flat_nu_ensemble_corr,
        "NaMaster": patch_flat_nmt_ensemble_corr,
        "PolSpice": patch_flat_pols_ensemble_corr,
    },
    "Medium mask": {
        "NaturalSpice": tr1_flat_nu_ensemble_corr,
        "NaMaster": tr1_flat_nmt_ensemble_corr,
        "PolSpice": tr1_flat_pols_ensemble_corr,
    },
    "Large mask": {
        "NaturalSpice": dr1_flat_nu_ensemble_corr,
        "NaMaster": dr1_flat_nmt_ensemble_corr,
        "PolSpice": dr1_flat_pols_ensemble_corr,
    },
}

def diagonality_score(corr):
    """1 means perfectly diagonal; smaller means more off-diagonal power."""
    n = corr.shape[0]
    offdiag = corr.copy()
    np.fill_diagonal(offdiag, 0.0)
    offdiag_energy = np.sum(abs(offdiag)) / (n * (n - 1))
    return offdiag_energy

ref_diag_score = diagonality_score(ref_corr)

method_colors = {
    "NaturalSpice": "#4477AA",
    "NaMaster": "#EE6677",
    "PolSpice": "#228833",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.subplots_adjust(top=0.82, wspace=0.2)  # leave room for shared legend

# store table values in ordered structure
diag_table = {}

for ax, (region, mats) in zip(axes, groups.items()):
    diag_table[region] = {}
    for label, mat in mats.items():
        # --- plotting logic unchanged ---
        eig_m = np.sort(np.linalg.eigvalsh(mat))[::-1]
        rel = np.abs(eig_m / den)
        x = np.arange(1, len(rel) + 1)
        ax.plot(x, rel, lw=1.5, label=label, color=method_colors[label])

        # --- new table metric: difference in diagonality score from reference ---
        mat_diag_score = diagonality_score(mat)
        diag_table[region][label] = mat_diag_score #abs(mat_diag_score - ref_diag_score)/ref_diag_score

    ax.axhline(1.0, color="k", ls="--", lw=1)
    ax.set_title(region, y=0.85)
    ax.set_xlabel("Eigenvalue index")
    ax.set_xscale("log")
    ax.set_yscale("symlog", linthresh=1e-3, linscale=1.0)
    ax.grid(alpha=0.25)

# Print LaTeX table
row_order = ["NaturalSpice", "NaMaster", "PolSpice"]

latex_lines = []
latex_lines.append(r"\begin{table}[ht]")
latex_lines.append(r"\centering")
latex_lines.append(r"\begin{tabular}{lccc}")
latex_lines.append(r"\hline")
latex_lines.append(r"Method & Patch & TR1 & DR1 \\")
latex_lines.append(r"\hline")

for label in row_order:
    patch_val = diag_table["Small mask"][label]
    tr1_val = diag_table["Medium mask"][label]
    dr1_val = diag_table["Large mask"][label]
    latex_lines.append(
        f"{label} & {patch_val:.4f} & {tr1_val:.4f} & {dr1_val:.4f} \\\\"
    )

latex_lines.append(r"\hline")
latex_lines.append(r"\end{tabular}")
latex_lines.append(
    r"\caption{Absolute difference in diagonality score, "
    r"$|D(R_{\rm method}) - D(R_{\rm full\text{-}sky})|$, "
    r"where $D(R)=1-\sum_{i\neq j} R_{ij}^2/[n(n-1)]$. "
    r"Smaller values indicate correlation matrices closer to the full-sky reference in diagonality.}"
)
latex_lines.append(r"\label{tab:diag_score_diff}")
latex_lines.append(r"\end{table}")

print("\n".join(latex_lines))

axes[0].set_ylabel(r"$|\lambda_{\rm deconvolved} / \lambda_{\rm full-sky}|$")

# shared legend (once for whole figure)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.98))

plt.show()
fig.savefig("./plots/eig_rel_diff_1x3_symlog.pdf", bbox_inches="tight")

In [ ]:
def get_xi2s(cls_data, cls_theory, covs, lmin=0, lmax=30):
    """Calculate the chi-squared value."""
    xi2s = {}
    for key in list(cls_data.keys()):
        a, b, i, j = key
        covkey = (a, b, a, b, i, j, i, j)
        cov = covs[covkey]
        cl_data = cls_data[key]
        cl_theory = cls_theory[key]
        if a == b == "POS":
            cl_data = cl_data.array[lmin:lmax]
            cl_theory = cl_theory.array[lmin:lmax]
            diff = cl_data - cl_theory
            cov = cov.array[lmin:lmax, lmin:lmax]
            invcov = np.linalg.pinv(cov)
            chol = cholesky_or_diagonalize(invcov)
            xi = diff @ chol
            xi2 = np.dot(xi, xi)
            xi2s[key] = heracles.Result(np.array([xi2/len(cl_data)]))
        elif a == b == "SHE":
            cl_data_ee = cl_data[0, 0, lmin:lmax]
            cl_data_eb = cl_data[0, 1, lmin:lmax]
            cl_data_bb = cl_data[1, 1, lmin:lmax]
            cl_theory_ee = cl_theory[0, 0, lmin:lmax]
            cl_theory_eb = cl_theory[0, 1, lmin:lmax]
            cl_theory_bb = cl_theory[1, 1, lmin:lmax]
            diff_ee = cl_data_ee - cl_theory_ee
            diff_eb = cl_data_eb - cl_theory_eb
            diff_bb = cl_data_bb - cl_theory_bb
            cov_ee = cov[0, 0, 0, 0, lmin:lmax, lmin:lmax]
            cov_eb = cov[0, 1, 0, 1, lmin:lmax, lmin:lmax]+1E-25
            cov_bb = cov[1, 1, 1, 1, lmin:lmax, lmin:lmax]+1E-25
            invcov_ee = np.linalg.pinv(cov_ee)
            invcov_eb = np.linalg.pinv(cov_eb)
            invcov_bb = np.linalg.pinv(cov_bb)
            chol_ee = cholesky_or_diagonalize(invcov_ee)
            chol_eb = cholesky_or_diagonalize(invcov_eb)
            chol_bb = cholesky_or_diagonalize(invcov_bb)
            xi_ee = diff_ee @ chol_ee
            xi_eb = diff_eb @ chol_eb
            xi_bb = diff_bb @ chol_bb
            xi2_ee = np.dot(xi_ee, xi_ee)
            xi2_eb = np.dot(xi_eb, xi_eb)
            xi2_bb = np.dot(xi_bb, xi_bb)
            xi2s[('E', 'E', i, j)] = heracles.Result(np.array([xi2_ee/len(cl_data_ee)]))
            xi2s[('E', 'B', i, j)] = heracles.Result(np.array([xi2_eb/len(cl_data_eb)]))
            xi2s[('B', 'B', i, j)] = heracles.Result(np.array([xi2_bb/len(cl_data_bb)]))
        elif a == "POS" and b == "SHE":
            cl_pe = cl_data[0, lmin:lmax]
            cl_pb = cl_data[1, lmin:lmax]
            cl_theory_pe = cl_theory[0, lmin:lmax]
            cl_theory_pb = cl_theory[1, lmin:lmax]
            diff_pe = cl_pe - cl_theory_pe
            diff_pb = cl_pb - cl_theory_pb
            cov_pe = cov[0, 0, lmin:lmax, lmin:lmax]
            cov_pb = cov[1, 1, lmin:lmax, lmin:lmax]+1E-25
            invcov_pe = np.linalg.pinv(cov_pe)
            invcov_pb = np.linalg.pinv(cov_pb)
            chol_pe = cholesky_or_diagonalize(invcov_pe)
            chol_pb = cholesky_or_diagonalize(invcov_pb)
            invcov_pe = np.linalg.pinv(cov_pe)
            invcov_pb = np.linalg.pinv(cov_pb)
            xi_pe = diff_pe @ chol_pe
            xi_pb = diff_pb @ chol_pb
            xi2_pe = np.dot(xi_pe, xi_pe)
            xi2_pb = np.dot(xi_pb, xi_pb)
            xi2s[('POS', 'E', i, j)] = heracles.Result(np.array([xi2_pe/len(cl_pe)]))
            xi2s[('POS', 'B', i, j)] = heracles.Result(np.array([xi2_pb/len(cl_pb)]))
        else:
            raise ValueError(f"Unknown key: {key}")
    return xi2s

def transpose_dict(data):
    """
    Same as transpose_nested_dict but returns NumPy arrays.
    """
    if not data:
        return {}

    inner_keys = {k for d in data.values() for k in d}

    return {
        key: np.array([data[outer][key] for outer in data])
        for key in inner_keys
    }

In [ ]:
min_ell = 14
patch_xi2s_fs_nu = {i: get_xi2s(patch_nu_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
patch_xi2s_fs_nmt = {i: get_xi2s(patch_nmt_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
patch_xi2s_fs_pols = {i: get_xi2s(patch_pols_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

min_ell=7
tr1_xi2s_fs_nu = {i: get_xi2s(tr1_nu_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
tr1_xi2s_fs_nmt = {i: get_xi2s(tr1_nmt_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
tr1_xi2s_fs_pols = {i: get_xi2s(tr1_pols_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

min_ell=5
dr1_xi2s_fs_nu = {i: get_xi2s(dr1_nu_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
dr1_xi2s_fs_nmt = {i: get_xi2s(dr1_nmt_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
dr1_xi2s_fs_pols = {i: get_xi2s(dr1_pols_cqs[i], full_sky_cqs[i], full_sky_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

_patch_xi2s_fs_nu = transpose_dict(patch_xi2s_fs_nu)
_patch_xi2s_fs_nmt = transpose_dict(patch_xi2s_fs_nmt)
_patch_xi2s_fs_pols = transpose_dict(patch_xi2s_fs_pols)

_tr1_xi2s_fs_nu = transpose_dict(tr1_xi2s_fs_nu)
_tr1_xi2s_fs_nmt = transpose_dict(tr1_xi2s_fs_nmt)
_tr1_xi2s_fs_pols = transpose_dict(tr1_xi2s_fs_pols)

_dr1_xi2s_fs_nu = transpose_dict(dr1_xi2s_fs_nu)
_dr1_xi2s_fs_nmt = transpose_dict(dr1_xi2s_fs_nmt)
_dr1_xi2s_fs_pols = transpose_dict(dr1_xi2s_fs_pols)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde

# ---------- config ----------
ci = 0.68          # central confidence interval mass (for the overlaid bar)
use_log_y = True   # compute the violin KDE in log10 space

mask_order = ["Small mask", "Medium mask", "Large mask"]
mask_data = {
    "Small mask": {
        "NaturalSpice": _patch_xi2s_fs_nu,
        "NaMaster": _patch_xi2s_fs_nmt,
        "PolSpice": _patch_xi2s_fs_pols,
    },
    "Medium mask": {
        "NaturalSpice": _tr1_xi2s_fs_nu,
        "NaMaster": _tr1_xi2s_fs_nmt,
        "PolSpice": _tr1_xi2s_fs_pols,
    },
    "Large mask": {
        "NaturalSpice": _dr1_xi2s_fs_nu,
        "NaMaster": _dr1_xi2s_fs_nmt,
        "PolSpice": _dr1_xi2s_fs_pols,
    },
}
fsky_map = {"Small mask": fsky_patch, "Medium mask": fsky_tr1, "Large mask": fsky_dr1}

field_keys = [("POS", "POS", 1, 1), ("POS", "E", 1, 1), ("E", "E", 1, 1)]
field_labels = ["PP", "PE", "EE"]

methods = ["NaturalSpice", "NaMaster", "PolSpice"]

# color now encodes the mask
mask_colors = {
    "Small mask": "#4477AA",
    "Medium mask": "#CCBB44",
    "Large mask": "#EE6677",
}

alpha = (1 - ci) / 2.0
q_lo, q_hi = 100 * alpha, 100 * (1 - alpha)


def plot_violin(ax, vals, xpos, half_w, color, log=True, n_grid=256):
    """Draw a single violin at x=xpos. KDE is built in log10 space when
    log=True so the shape is correct on a log y-axis."""
    vals = np.asarray(vals, dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    if log:
        vals = vals[vals > 0]
    if vals.size == 0:
        return

    data = np.log10(vals) if log else vals

    # degenerate distribution -> just mark the level
    if data.size < 3 or np.allclose(data, data[0]):
        y = 10 ** np.median(data) if log else np.median(data)
        ax.plot([xpos - half_w, xpos + half_w], [y, y], color=color, lw=1.2)
        return

    kde = gaussian_kde(data)
    lo, hi = data.min(), data.max()
    pad = 0.06 * (hi - lo)
    grid = np.linspace(lo - pad, hi + pad, n_grid)
    raw = kde(grid)

    dens = raw / raw.max() * half_w          # scale every violin to a fixed width
    yy = 10 ** grid if log else grid
    ax.fill_betweenx(yy, xpos - dens, xpos + dens,
                     facecolor=color, alpha=0.7, edgecolor=color, lw=0.8)

    # overlay mode + 68% interval (carries over the old bar/errorbar info)
    mode = grid[np.argmax(raw)]
    lo_p, hi_p = np.percentile(data, [q_lo, q_hi])
    to_y = (lambda v: 10 ** v) if log else (lambda v: v)
    ax.plot([xpos, xpos], [to_y(lo_p), to_y(hi_p)], color="black", lw=1.0, alpha=0.8)
    ax.plot(xpos, to_y(mode), marker="o", ms=3, color="black", zorder=5)


fig, axes = plt.subplots(len(methods), 1, figsize=(9, 10), sharex=True)
fig.subplots_adjust(hspace=0.05, top=0.92)

x = np.arange(len(field_labels))
width = 0.18                     # spacing between the three masks
half_w = width * 0.85 / 2.0      # half-width of each violin

for ax, method in zip(axes, methods):
    for i, mask_name in enumerate(mask_order):
        fsky = fsky_map[mask_name]
        xbase = x + i * width - 1.0 * width   # centre the group of three
        for j, key in enumerate(field_keys):
            vals = np.asarray(mask_data[mask_name][method][key], dtype=float).ravel() * fsky
            plot_violin(ax, vals, xbase[j], half_w,
                        color=mask_colors[mask_name], log=use_log_y)

    ax.set_yscale("log")
    ax.set_ylim(5e-2, 1e10)
    ax.set_ylabel(r"$\tilde{\chi}_{\rm FS}^2$")
    ax.set_title(method, y=0.85)
    ax.grid(axis="y", alpha=0.25, which="major")

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(field_labels)

# Shared legend (proxy patches, since violins don't produce handles)
handles = [Patch(facecolor=mask_colors[mk], alpha=0.7, label=mk) for mk in mask_order]
fig.legend(handles, mask_order, loc="upper center", ncols=len(mask_order),
           bbox_to_anchor=(0.5, 0.98), frameon=False)

fig.savefig(f"./plots/chi2_FS_rcond_{rcond}_comp.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
lgrid[7]

In [ ]:
min_ell = 14
patch_xi2s_nu = {i: get_xi2s(patch_nu_cqs[i], full_sky_cqs[i], patch_nu_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
patch_xi2s_nmt = {i: get_xi2s(patch_nmt_cqs[i], full_sky_cqs[i], patch_nmt_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
patch_xi2s_pols = {i: get_xi2s(patch_pols_cqs[i], full_sky_cqs[i], patch_pols_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

min_ell=7
tr1_xi2s_nu = {i: get_xi2s(tr1_nu_cqs[i], full_sky_cqs[i], tr1_nu_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
tr1_xi2s_nmt = {i: get_xi2s(tr1_nmt_cqs[i], full_sky_cqs[i], tr1_nmt_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
tr1_xi2s_pols = {i: get_xi2s(tr1_pols_cqs[i], full_sky_cqs[i], tr1_pols_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

min_ell=5
dr1_xi2s_nu = {i: get_xi2s(dr1_nu_cqs[i], full_sky_cqs[i], dr1_nu_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
dr1_xi2s_nmt = {i: get_xi2s(dr1_nmt_cqs[i], full_sky_cqs[i], dr1_nmt_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}
dr1_xi2s_pols = {i: get_xi2s(dr1_pols_cqs[i], full_sky_cqs[i], tr1_pols_ensemble_cov, lmin=min_ell) for i in list(full_sky_cqs.keys())}

_patch_xi2s_nu = transpose_dict(patch_xi2s_nu)
_patch_xi2s_nmt = transpose_dict(patch_xi2s_nmt)
_patch_xi2s_pols = transpose_dict(patch_xi2s_pols)

_tr1_xi2s_nu = transpose_dict(tr1_xi2s_nu)
_tr1_xi2s_nmt = transpose_dict(tr1_xi2s_nmt)
_tr1_xi2s_pols = transpose_dict(tr1_xi2s_pols)

_dr1_xi2s_nu = transpose_dict(dr1_xi2s_nu)
_dr1_xi2s_nmt = transpose_dict(dr1_xi2s_nmt)
_dr1_xi2s_pols = transpose_dict(dr1_xi2s_pols)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde

# ---------- config ----------
ci = 0.68          # central confidence interval mass (for the overlaid bar)
use_log_y = True   # compute the violin KDE in log10 space

mask_order = ["Small mask", "Medium mask", "Large mask"]
mask_data = {
    "Small mask": {
        "NaturalSpice": _patch_xi2s_nu,
        "NaMaster": _patch_xi2s_nmt,
        "PolSpice": _patch_xi2s_pols,
    },
    "Medium mask": {
        "NaturalSpice": _tr1_xi2s_nu,
        "NaMaster": _tr1_xi2s_nmt,
        "PolSpice": _tr1_xi2s_pols,
    },
    "Large mask": {
        "NaturalSpice": _dr1_xi2s_nu,
        "NaMaster": _dr1_xi2s_nmt,
        "PolSpice": _dr1_xi2s_pols,
    },
}
fsky_map = {"Small mask": fsky_patch, "Medium mask": fsky_tr1, "Large mask": fsky_dr1}

field_keys = [
    ("POS", "POS", 1, 1), ("POS", "E", 1, 1), ("POS", "B", 1, 1),
    ("E", "E", 1, 1), ("E", "B", 1, 1), ("B", "B", 1, 1),
]
field_labels = ["PP", "PE", "PB", "EE", "EB", "BB"]

methods = ["NaturalSpice", "NaMaster", "PolSpice"]

# color now encodes the mask
mask_colors = {
    "Small mask": "#4477AA",
    "Medium mask": "#CCBB44",
    "Large mask": "#EE6677",
}

alpha = (1 - ci) / 2.0
q_lo, q_hi = 100 * alpha, 100 * (1 - alpha)


def plot_violin(ax, vals, xpos, half_w, color, log=True, n_grid=256):
    """Draw a single violin at x=xpos. KDE is built in log10 space when
    log=True so the shape is correct on a log y-axis."""
    vals = np.asarray(vals, dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    if log:
        vals = vals[vals > 0]
    if vals.size == 0:
        return

    data = np.log10(vals) if log else vals

    # degenerate distribution -> just mark the level
    if data.size < 3 or np.allclose(data, data[0]):
        y = 10 ** np.median(data) if log else np.median(data)
        ax.plot([xpos - half_w, xpos + half_w], [y, y], color=color, lw=1.2)
        return

    kde = gaussian_kde(data)
    lo, hi = data.min(), data.max()
    pad = 0.06 * (hi - lo)
    grid = np.linspace(lo - pad, hi + pad, n_grid)
    raw = kde(grid)

    dens = raw / raw.max() * half_w          # scale every violin to a fixed width
    yy = 10 ** grid if log else grid
    ax.fill_betweenx(yy, xpos - dens, xpos + dens,
                     facecolor=color, alpha=0.7, edgecolor=color, lw=0.8)

    # overlay mode + 68% interval (carries over the old bar/errorbar info)
    mode = grid[np.argmax(raw)]
    lo_p, hi_p = np.percentile(data, [q_lo, q_hi])
    to_y = (lambda v: 10 ** v) if log else (lambda v: v)
    ax.plot([xpos, xpos], [to_y(lo_p), to_y(hi_p)], color="black", lw=1.0, alpha=0.8)
    ax.plot(xpos, to_y(mode), marker="o", ms=3, color="black", zorder=5)


fig, axes = plt.subplots(len(methods), 1, figsize=(9, 10), sharex=True)
fig.subplots_adjust(hspace=0.05, top=0.92)

x = np.arange(len(field_labels))
width = 0.18                     # spacing between the three masks
half_w = width * 0.85 / 2.0      # half-width of each violin

for ax, method in zip(axes, methods):
    for i, mask_name in enumerate(mask_order):
        fsky = 1  # fsky_map[mask_name]
        xbase = x + i * width - 1.0 * width   # centre the group of three
        for j, key in enumerate(field_keys):
            vals = np.asarray(mask_data[mask_name][method][key], dtype=float).ravel() * fsky
            plot_violin(ax, vals, xbase[j], half_w,
                        color=mask_colors[mask_name], log=use_log_y)

    ax.set_yscale("log")
    ax.set_ylim(5e-2, 5e2)
    ax.set_ylabel(r"$\tilde{\chi}^2$")
    ax.set_title(method, y=0.85)
    ax.grid(axis="y", alpha=0.25, which="major")

axes[-1].set_xticks(x)
axes[-1].set_xticklabels(field_labels)

# Shared legend (proxy patches, since violins don't produce handles)
handles = [Patch(facecolor=mask_colors[mk], alpha=0.7, label=mk) for mk in mask_order]
fig.legend(handles, mask_order, loc="upper center", ncols=len(mask_order),
           bbox_to_anchor=(0.5, 0.98), frameon=False)

fig.savefig(f"./plots/chi2_rcond_{rcond}_comp.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
from scipy.stats import gaussian_kde

# Reuse the SAME structures you defined for the violin figure:
#   mask_data[mask][method][field_key] -> 1D array of chi^2 over simulations
#   mask_order, methods, field_keys, field_labels
# (paste this cell after those are in scope)


def kde_mode(x, n_grid=512):
    x = np.asarray(x, dtype=float).ravel()
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    if x.size < 3 or np.allclose(x, x[0]):
        return float(np.median(x))
    kde = gaussian_kde(x)
    lo, hi = np.min(x), np.max(x)
    if lo == hi:
        return float(lo)
    grid = np.linspace(lo, hi, n_grid)
    return float(grid[np.argmax(kde(grid))])


def _summary(vals, central, q_lo, q_hi):
    """central value + asymmetric (lo, hi) errors from percentiles."""
    vals = np.asarray(vals, dtype=float).ravel()
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return None
    p_lo, p_hi = np.percentile(vals, [q_lo, q_hi])
    if central == "mode":
        c = kde_mode(vals)
    elif central == "mean":
        c = float(np.mean(vals))
    elif central == "median":
        c = float(np.median(vals))
    else:
        raise ValueError(f"unknown central='{central}'")
    return c, max(c - p_lo, 0.0), max(p_hi - c, 0.0)


def make_chi2_table(
    mask_data,
    mask_order,
    methods,
    field_keys,
    field_labels,
    fsky=1.0,                # scalar, or a dict {mask: fsky}; values are multiplied in
    central="mode",          # "mode" (matches the violin overlay), "mean", or "median"
    ci=0.68,
    fmt="{:.2f}",
    quantity=r"\chi^2",
    caption=(
        "Reduced goodness of fit of each unmixing algorithm computed with respect "
        "to a full-sky angular power spectrum, using the covariance of the "
        "deconvolved angular power spectra. Different rows show different masks "
        "while different columns show different auto- and cross-correlations. We "
        "observe that \\texttt{NaMaster} and real-space \\texttt{NaturalSpice} are "
        "the only two algorithms consistently returning reduced goodness of fit of "
        "order one regardless of the mask applied to the data or the correlation "
        "considered."
    ),
    label="tab:chi2",
):
    alpha = (1 - ci) / 2.0
    q_lo, q_hi = 100 * alpha, 100 * (1 - alpha)

    ncol = len(field_keys)
    colspec = "ll" + "c" * ncol

    out = []
    out.append(r"\begin{table*}")
    out.append(r"\centering")
    out.append(r"\renewcommand{\arraystretch}{1.4}   % <-- adjust this number to taste")
    out.append(r"\begin{tabular}{" + colspec + "}")
    out.append(r"\hline")
    out.append(r"\multicolumn{" + str(ncol + 2) + r"}{c}{$" + quantity + r"$} \\")
    out.append(r"\hline")
    out.append("Mask & Method & " + " & ".join(field_labels) + r" \\")
    out.append(r"\hline")

    for mask_name in mask_order:
        f = fsky[mask_name] if isinstance(fsky, dict) else fsky

        for mi, method in enumerate(methods):
            cells = []
            for key in field_keys:
                vals = np.asarray(mask_data[mask_name][method][key], dtype=float).ravel() * f
                s = _summary(vals, central, q_lo, q_hi)
                if s is None:
                    cells.append(r"$-$")
                else:
                    c, e_lo, e_hi = s
                    cells.append(
                        "$" + fmt.format(c)
                        + "_{-" + fmt.format(e_lo) + "}"
                        + "^{+" + fmt.format(e_hi) + "}$"
                    )

            first = (
                r"\multirow{" + str(len(methods)) + r"}{*}{" + mask_name + "}"
                if mi == 0 else ""
            )
            out.append(f"{first} & {method} & " + " & ".join(cells) + r" \\")

        out.append(r"\hline")  # rule after every mask block (incl. the last)

    out.append(r"\end{tabular}")
    out.append(r"\caption{" + caption + "}")
    out.append(r"\label{" + label + "}")
    out.append(r"\end{table*}")
    return "\n".join(out)


if __name__ == "__main__":
    # fsky = 1 here, matching the violin figure (fsky_map line was commented out)
    tex = make_chi2_table(
        mask_data, mask_order, methods, field_keys, field_labels,
        fsky=1.0, central="mode",
    )
    print(tex)